**M2 - Análise Exploratória de Dados - Gustavo Lobato Campos e Rafael Vinicius Tayette da Nobrega**

**Fonte dos dados:** [Kaggle - Solar Power Generation Data (anikannal)](https://www.kaggle.com/datasets/anikannal/solar-power-generation-data)

Data de acesso: 02/07/2026

Este notebook implementa a análise exploratória de dados (EDA) de duas usinas solares fotovoltaicas (Planta 1 e Planta 2), cada uma com dois arquivos: dados de geração dos inversores (`Generation_Data.csv`) e dados do sensor meteorológico (`Weather_Sensor_Data.csv`).

Estrutura do notebook: 
1. Configuração inicial 
2. Planta 1: contexto, qualidade, distribuições e relações
3. Planta 2: contexto, qualidade, distribuições e relações
4. Comparação entre as Plantas 1 e 2


**1. Configuração inicial:** 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style = "whitegrid", palette = "viridis")
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.titleweight'] = 'bold'

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

RANDOM_STATE = 42
TARGET = 'AC_POWER'   # variável alvo: potência AC gerada pelo inversor (kW)

def resumo_ausentes(df, nome):
    # Tabela de valores ausentes por coluna, com taxa (%) e decisão sugerida
    n = len(df)
    tab = pd.DataFrame({
        'coluna': df.columns,
        'n_ausentes': df.isna().sum().values,
    })
    tab['taxa_%'] = (tab['n_ausentes'] / n * 100).round(3)
    def decisao(taxa):
        if taxa == 0:
            return 'Manter (sem ausentes)'
        elif taxa < 5:
            return 'Imputar (interpolação temporal)'
        elif taxa < 30:
            return 'Avaliar imputação por série temporal/vizinhos'
        else:
            return 'Considerar remoção da coluna'
    tab['decisão'] = tab['taxa_%'].apply(decisao)
    tab.insert(0, 'dataset', nome)
    return tab.reset_index(drop = True)

def resumo_contexto(df, nome, dt_col = 'DATE_TIME'):
    print(f"--- {nome} ---")
    print(f"Nº de registros : {len(df):,}".replace(',', '.'))
    print(f"Nº de variáveis : {df.shape[1]}")
    print(f"Variáveis       : {list(df.columns)}")
    print(f"Período         : {df[dt_col].min()} até {df[dt_col].max()}")
    print()


**2. Planta 1:**

2.1 Contexto: 

Arquivos utilizados: `Plant_1_Generation_Data.csv` (dados de geração por inversor, granularidade de 15 min) e `Plant_1_Weather_Sensor_Data.csv` (dados do sensor meteorológico da Planta 1).

Data de acesso: 02/07/2026

Período coberto: 15/05/2020 a 17/06/2020 (aproximadamente 34 dias)

In [ ]:
# Carregamento dos dados da Planta 1
gen1 = pd.read_csv('Plant_1_Generation_Data.csv')
wth1 = pd.read_csv('Plant_1_Weather_Sensor_Data.csv')

# Padronização das datas dos dois arquivo
gen1['DATE_TIME'] = pd.to_datetime(gen1['DATE_TIME'], format='%d-%m-%Y %H:%M')
wth1['DATE_TIME'] = pd.to_datetime(wth1['DATE_TIME'], format='%Y-%m-%d %H:%M:%S')

resumo_contexto(gen1, 'Plant_1_Generation_Data.csv')
resumo_contexto(wth1, 'Plant_1_Weather_Sensor_Data.csv')

print(f"Nº de inversores (SOURCE_KEY) na Planta 1: {gen1['SOURCE_KEY'].nunique()}")


In [ ]:
# Merge dos dados de geração + clima pela data/hora e planta, formando o dataframe da Planta 1
df1 = pd.merge(gen1, wth1.drop(columns = ['SOURCE_KEY']), on = ['DATE_TIME', 'PLANT_ID'], how = 'left')

# Cópia do estado ANTES de qualquer imputação — usada na tabela de qualidade (ausentes)
df1_pre_imputacao = df1.copy()

print(f"Base combinada Planta 1: {df1.shape[0]:,} registros x {df1.shape[1]} colunas".replace(',', '.'))
df1.head()

2.1.1 Estatísticas descritivas da base combinada da Planta 1: 

In [ ]:
desc1 = df1.describe().T
desc1


2.2 Qualidade dos dados: 

In [ ]:
# Ausentes: arquivos originais + base combinada (estado ANTES da imputação)
# O merge pode gerar ausentes se algum horário do inversor não tiver leitura correspondente no sensor meteorológico
tab_ausentes_1 = pd.concat([
    resumo_ausentes(gen1, 'Plant_1_Generation_Data'),
    resumo_ausentes(wth1, 'Plant_1_Weather_Sensor_Data'),
    resumo_ausentes(df1_pre_imputacao, 'Plant_1_Combinado (merge)'),
], ignore_index = True)
tab_ausentes_1

In [ ]:
# Duplicatas: antes e depois (linhas 100% idênticas em todas as colunas)
dup_gen1_antes = gen1.duplicated().sum()
dup_wth1_antes = wth1.duplicated().sum()

gen1_dedup = gen1.drop_duplicates()
wth1_dedup = wth1.drop_duplicates()

dup_gen1_depois = gen1_dedup.duplicated().sum()
dup_wth1_depois = wth1_dedup.duplicated().sum()

# Duplicatas: mesmo inversor (SOURCE_KEY) e mesmo DATE_TIME
dup_logica_gen1 = gen1.duplicated(subset=['DATE_TIME', 'SOURCE_KEY']).sum()

tab_dup_1 = pd.DataFrame({
    'dataset': ['Plant_1_Generation_Data', 'Plant_1_Weather_Sensor_Data'],
    'duplicatas_antes': [dup_gen1_antes, dup_wth1_antes],
    'duplicatas_depois': [dup_gen1_depois, dup_wth1_depois],
    'duplicatas_logicas_(DATE_TIME+SOURCE_KEY)': [dup_logica_gen1, np.nan],
})
tab_dup_1


In [ ]:
n_aus_merge_1 = tab_ausentes_1.query("dataset == 'Plant_1_Combinado (merge)'")['n_ausentes'].sum()
print("Avaliação da qualidade da Planta 1:")
print(" - Ausentes nos arquivos originais: 0 em todas as colunas -> manter, sem necessidade de imputação.")
if n_aus_merge_1 == 0:
    print(" - Após o merge geração + clima: nenhum ausente introduzido -> base combinada íntegra.")
else:
    print(f" - Após o merge geração + clima: {int(n_aus_merge_1)} ausente(s) nas colunas climáticas "
          f"-> imputação por interpolação temporal.")
print(f" - Duplicatas exatas: {dup_gen1_antes} (geração) e {dup_wth1_antes} (clima) -> nenhuma remoção necessária.")

Implementação da imputação por interpolação temporal na base combinada da Planta 1: interpolação linear ordenada pelo tempo, usando os pontos vizinhos:

In [ ]:
cols_clima_1 = ['AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']
n_ausentes_antes_1 = df1[cols_clima_1].isna().sum().sum()

df1 = df1.sort_values('DATE_TIME')
df1[cols_clima_1] = df1[cols_clima_1].interpolate(method = 'linear', limit_direction = 'both')

n_ausentes_depois_1 = df1[cols_clima_1].isna().sum().sum()
print(f"Ausentes nas colunas climáticas antes da imputação : {n_ausentes_antes_1}")
print(f"Ausentes nas colunas climáticas depois da imputação: {n_ausentes_depois_1}")


2.3 Distribuições:

In [ ]:
features_1 = ['DC_POWER', 'AC_POWER', 'DAILY_YIELD', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']

fig, axes = plt.subplots(2, 3, figsize = (15, 8))
for ax, col in zip(axes.ravel(), features_1):
    sns.histplot(df1[col].dropna(), bins = 40, kde = True, ax = ax, color = '#2b7a78')
    ax.set_title(col)
    ax.set_xlabel('')
fig.suptitle('Planta 1 - Distribuição das principais variáveis', fontsize = 14, y = 1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Distribuição do alvo AC_POWER e taxa de desbalanceamento
# Alvo contínuo com forte concentração em zero (períodos noturnos sem irradiação)
# Avaliação do desbalanceamento: binarizando a Geração AC_POWER > 0 e sem geração AC_POWER == 0

fig, axes = plt.subplots(1, 2, figsize = (13, 5))
sns.histplot(df1[TARGET], bins = 50, kde = True, ax = axes[0], color = '#c1440e')
axes[0].set_title('Distribuição de AC_POWER')
axes[0].set_ylabel('Contagem')
axes[0].set_xlabel('AC_POWER (kW)')

classe_1 = np.where(df1[TARGET] > 0, 'Geração (> 0)', 'Sem geração (= 0)')
contagem_1 = pd.Series(classe_1).value_counts()
taxa_desbal_1 = contagem_1 / contagem_1.sum() * 100

pct_sem_geracao_1 = float((df1[TARGET] == 0).mean() * 100)
pct_com_geracao_1 = 100 - pct_sem_geracao_1

sns.barplot(x = contagem_1.index, y = contagem_1.values, hue = contagem_1.index, ax = axes[1],
            palette=['#2b7a78', '#c1440e'], legend = False)
axes[1].set_title('Balanceamento do alvo binarizado')
axes[1].set_ylabel('Nº de registros')
axes[1].set_xlabel('Taxa de desbalanceamento')
for i, v in enumerate(contagem_1.values):
    axes[1].text(i, v, f"{v:,}\n({taxa_desbal_1.values[i]:.1f}%)".replace(',', '.'), ha = 'center', va = 'bottom')

plt.ylim(0, 42500)
plt.tight_layout()
plt.show()

print(f"Taxa de desbalanceamento (Planta 1): {taxa_desbal_1.round(4).to_dict()}")
razao_1 = contagem_1.max() / contagem_1.min()
print(f"Razão entre classe majoritária/minoritária: {razao_1:.3f} : 1")


2.4 Relações entre variáveis:

In [ ]:
cols_num_1 = ['DC_POWER', 'AC_POWER', 'DAILY_YIELD', 'TOTAL_YIELD',
              'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']
corr_1 = df1[cols_num_1].corr()

plt.figure(figsize = (8, 6))
sns.heatmap(corr_1, annot = True, fmt = '.2f', cmap = 'coolwarm', vmin = -1, vmax = 1, square = True)
plt.title('Planta 1 - Matriz de correlação (Pearson)')
plt.tight_layout()
plt.show()


In [ ]:
top5_corr_1 = corr_1[TARGET].drop(TARGET).abs().sort_values(ascending = False).head(10)
top5_corr_1_tab = pd.DataFrame({
    'variável': top5_corr_1.index,
    'correlação_com_AC_POWER': corr_1[TARGET].loc[top5_corr_1.index].values,
    '|correlação|': top5_corr_1.values,
}).reset_index(drop = True)
top5_corr_1_tab


In [ ]:
# Scatter plots dos pares mais críticos com maiores correlações com o alvo
N_PARES_CRITICOS = 4
pares_criticos_1 = list(top5_corr_1_tab['variável'].head(N_PARES_CRITICOS))
amostra_1 = df1.sample(min(4000, len(df1)), random_state = RANDOM_STATE)

fig, axes = plt.subplots(2, 3, figsize = (16, 9))
axes = axes.ravel()
for ax, col in zip(axes, pares_criticos_1):
    sns.scatterplot(data = amostra_1, x = col, y = TARGET, alpha = 0.3, s = 12, ax = ax, color = '#2b7a78')
    ax.set_title(f'{col}  vs  {TARGET}\n(r = {corr_1.loc[col, TARGET]:.3f})')
for ax in axes[len(pares_criticos_1):]:
    ax.axis('off')
fig.suptitle('Planta 1 - Scatter plots dos pares críticos com o alvo', y = 1.01, fontsize = 13)
plt.tight_layout()
plt.show()


**3. Planta 2:**

3.1 Contexto: 

Arquivos utilizados: `Plant_2_Generation_Data.csv` (dados de geração por inversor, granularidade de 15 min) e `Plant_2_Weather_Sensor_Data.csv` (dados do sensor meteorológico da Planta 2).

Data de acesso: 02/07/2026

Período coberto: 15/05/2020 a 17/06/2020 (aproximadamente 34 dias)

In [ ]:
# Carregamento dos dados da Planta 2
gen2 = pd.read_csv('Plant_2_Generation_Data.csv')
wth2 = pd.read_csv('Plant_2_Weather_Sensor_Data.csv')

gen2['DATE_TIME'] = pd.to_datetime(gen2['DATE_TIME'], format='%Y-%m-%d %H:%M:%S')
wth2['DATE_TIME'] = pd.to_datetime(wth2['DATE_TIME'], format='%Y-%m-%d %H:%M:%S')

resumo_contexto(gen2, 'Plant_2_Generation_Data.csv')
resumo_contexto(wth2, 'Plant_2_Weather_Sensor_Data.csv')

print(f"Nº de inversores (SOURCE_KEY) na Planta 2: {gen2['SOURCE_KEY'].nunique()}")


In [ ]:
df2 = pd.merge(gen2, wth2.drop(columns = ['SOURCE_KEY']), on = ['DATE_TIME', 'PLANT_ID'], how = 'left')
print(f"Base combinada Planta 2: {df2.shape[0]:,} registros x {df2.shape[1]} colunas".replace(',', '.'))
df2.head()


3.1.1 Estatísticas descritivas da base combinada da Planta 2: 

In [ ]:
desc2 = df2.describe().T
desc2


3.2 Qualidade dos dados: 

In [ ]:
tab_ausentes_2 = pd.concat([
    resumo_ausentes(gen2, 'Plant_2_Generation_Data'),
    resumo_ausentes(wth2, 'Plant_2_Weather_Sensor_Data'),
    resumo_ausentes(df2,  'Plant_2_Combinado (merge)'),
], ignore_index = True)
tab_ausentes_2


In [ ]:
dup_gen2_antes = gen2.duplicated().sum()
dup_wth2_antes = wth2.duplicated().sum()

gen2_dedup = gen2.drop_duplicates()
wth2_dedup = wth2.drop_duplicates()

dup_gen2_depois = gen2_dedup.duplicated().sum()
dup_wth2_depois = wth2_dedup.duplicated().sum()

dup_logica_gen2 = gen2.duplicated(subset=['DATE_TIME', 'SOURCE_KEY']).sum()

tab_dup_2 = pd.DataFrame({
    'dataset': ['Plant_2_Generation_Data', 'Plant_2_Weather_Sensor_Data'],
    'duplicatas_antes': [dup_gen2_antes, dup_wth2_antes],
    'duplicatas_depois': [dup_gen2_depois, dup_wth2_depois],
    'duplicatas_logicas_(DATE_TIME+SOURCE_KEY)': [dup_logica_gen2, np.nan],
})
tab_dup_2


In [ ]:
print("Avaliação da qualidade da Planta 2:")
print(f" - Ausentes nos arquivos originais: 0 em todas as colunas -> manter, sem necessidade de imputação.")
if tab_ausentes_2.query("dataset == 'Plant_2_Combinado (merge)'")['n_ausentes'].sum() == 0:
    print(" - Após o merge geração + clima: nenhum ausente introduzido -> base combinada íntegra.")
else:
    print(" - Após o merge geração + clima: há ausentes -> avaliar imputação por interpolação temporal.")
print(f" - Duplicatas exatas: {dup_gen2_antes} (geração) e {dup_wth2_antes} (clima) -> nenhuma remoção necessária.")


Implementação da imputação por interpolação temporal na base combinada da Planta 2: interpolação linear ordenada pelo tempo, usando os pontos vizinhos:

In [ ]:
cols_clima_2 = ['AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']
n_ausentes_antes_2 = df2[cols_clima_2].isna().sum().sum()

if n_ausentes_antes_2 > 0:
    df2 = df2.sort_values('DATE_TIME')
    df2[cols_clima_2] = df2[cols_clima_2].interpolate(method = 'linear', limit_direction = 'both')

n_ausentes_depois_2 = df2[cols_clima_2].isna().sum().sum()
print(f"Ausentes nas colunas climáticas antes da imputação : {n_ausentes_antes_2}")
print(f"Ausentes nas colunas climáticas depois da imputação: {n_ausentes_depois_2}")


3.3 Distribuições: 

In [ ]:
features_2 = ['DC_POWER', 'AC_POWER', 'DAILY_YIELD', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']

fig, axes = plt.subplots(2, 3, figsize = (15, 8))
for ax, col in zip(axes.ravel(), features_2):
    sns.histplot(df2[col].dropna(), bins = 40, kde = True, ax = ax, color = '#5b3a9e')
    ax.set_title(col)
    ax.set_xlabel('')
fig.suptitle('Planta 2 - Distribuição das principais variáveis', fontsize = 14, y = 1.02)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize = (13, 5))
sns.histplot(df2[TARGET], bins = 50, kde = True, ax = axes[0], color = '#c1440e')
axes[0].set_title('Distribuição de AC_POWER')
axes[0].set_ylabel('Contagem')
axes[0].set_xlabel('AC_POWER (kW)')

classe_2 = np.where(df2[TARGET] > 0, 'Geração (> 0)', 'Sem geração (= 0)')
contagem_2 = pd.Series(classe_2).value_counts()
taxa_desbal_2 = contagem_2 / contagem_2.sum() * 100

pct_sem_geracao_2 = float((df2[TARGET] == 0).mean() * 100)
pct_com_geracao_2 = 100 - pct_sem_geracao_2

sns.barplot(x = contagem_2.index, y = contagem_2.values, hue = contagem_2.index, ax = axes[1],
            palette=['#5b3a9e', '#c1440e'], legend = False)
axes[1].set_title('Balanceamento do alvo binarizado')
axes[1].set_ylabel('Nº de registros')
axes[1].set_xlabel('Taxa de desbalanceamento')

for i, v in enumerate(contagem_2.values):
    axes[1].text(i, v, f"{v:,}\n({taxa_desbal_2.values[i]:.1f}%)".replace(',', '.'), ha = 'center', va = 'bottom')

plt.ylim(0, 42500)
plt.tight_layout()
plt.show()

print(f"Taxa de desbalanceamento (Planta 2): {taxa_desbal_2.round(4).to_dict()}")
razao_2 = contagem_2.max() / contagem_2.min()
print(f"Razão entre classe majoritária/minoritária: {razao_2:.3f} : 1")


3.4 Relações entre variáveis: 

In [ ]:
cols_num_2 = ['DC_POWER', 'AC_POWER', 'DAILY_YIELD', 'TOTAL_YIELD',
              'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']
corr_2 = df2[cols_num_2].corr()

plt.figure(figsize = (8, 6))
sns.heatmap(corr_2, annot = True, fmt = '.2f', cmap = 'coolwarm', vmin = -1, vmax = 1, square = True)
plt.title('Planta 2 - Matriz de correlação (Pearson)')
plt.tight_layout()
plt.show()


In [ ]:
top5_corr_2 = corr_2[TARGET].drop(TARGET).abs().sort_values(ascending = False).head(10)
top5_corr_2_tab = pd.DataFrame({
    'variável': top5_corr_2.index,
    'correlação_com_AC_POWER': corr_2[TARGET].loc[top5_corr_2.index].values,
    '|correlação|': top5_corr_2.values,
}).reset_index(drop = True)
top5_corr_2_tab


In [ ]:
pares_criticos_2 = list(top5_corr_2_tab['variável'].head(N_PARES_CRITICOS))
amostra_2 = df2.sample(min(4000, len(df2)), random_state = RANDOM_STATE)

fig, axes = plt.subplots(2, 3, figsize = (16, 9))
axes = axes.ravel()
for ax, col in zip(axes, pares_criticos_2):
    sns.scatterplot(data = amostra_2, x = col, y = TARGET, alpha = 0.3, s = 12, ax = ax, color = '#5b3a9e')
    ax.set_title(f'{col}  vs  {TARGET}\n(r = {corr_2.loc[col, TARGET]:.3f})')
for ax in axes[len(pares_criticos_2):]:
    ax.axis('off')
fig.suptitle('Planta 2 - Scatter plots dos pares críticos com o alvo', y = 1.01, fontsize = 13)
plt.tight_layout()
plt.show()


**4. Comparação entre Plantas 1 e 2**

In [ ]:
contexto_comp = pd.DataFrame({
    'Métrica': ['Nº de registros (geração)', 'Nº de registros (clima)', 'Nº de inversores',
                'Início do período', 'Fim do período', 'Nº de colunas (base combinada)'],
    'Planta 1': [len(gen1), len(wth1), gen1['SOURCE_KEY'].nunique(),
                 str(df1['DATE_TIME'].min()), str(df1['DATE_TIME'].max()), df1.shape[1]],
    'Planta 2': [len(gen2), len(wth2), gen2['SOURCE_KEY'].nunique(),
                 str(df2['DATE_TIME'].min()), str(df2['DATE_TIME'].max()), df2.shape[1]],
})
contexto_comp


4.1 Estatísticas descritivas comparadas: 

In [ ]:
desc_comp = pd.concat(
    {'Planta 1': desc1, 'Planta 2': desc2},
    axis = 1
)
desc_comp


4.2 Qualidade comparada (ausentes e duplicatas): 

In [ ]:
qualidade_comp = pd.DataFrame({
    'Métrica': ['Ausentes totais (geração)', 'Ausentes totais (clima)',
                'Duplicatas exatas (geração)', 'Duplicatas exatas (clima)',
                'Ausentes após merge geração + clima'],
    'Planta 1': [gen1.isna().sum().sum(), wth1.isna().sum().sum(),
                 dup_gen1_antes, dup_wth1_antes, df1.isna().sum().sum()],
    'Planta 2': [gen2.isna().sum().sum(), wth2.isna().sum().sum(),
                 dup_gen2_antes, dup_wth2_antes, df2.isna().sum().sum()],
})
qualidade_comp


4.3 Distribuições comparadas: 

In [ ]:
fig, axes = plt.subplots(1, 2, figsize = (13, 5), sharey = True)
sns.histplot(df1[TARGET], bins = 50, kde = True, ax = axes[0], color = '#2b7a78')
axes[0].set_title('Planta 1 - AC_POWER')
sns.histplot(df2[TARGET], bins = 50, kde = True, ax = axes[1], color = '#5b3a9e')
axes[1].set_title('Planta 2 - AC_POWER')
fig.suptitle('Comparação da distribuição do alvo (AC_POWER)', y = 1.03, fontsize = 13)
plt.tight_layout()
plt.show()

balanc_comp = pd.DataFrame({
    'Planta': ['Planta 1', 'Planta 2'],
    '% Sem geração (= 0)': [pct_sem_geracao_1, pct_sem_geracao_2],
    '% Geração (> 0)': [pct_com_geracao_1, pct_com_geracao_2],
    'Razão majoritário e minoritário': [razao_1, razao_2],
}).round(3)
balanc_comp


In [ ]:
fig, axes = plt.subplots(2, 3, figsize = (15, 8), sharex = False)
feats_cmp = ['DC_POWER', 'AC_POWER', 'DAILY_YIELD', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']
for ax, col in zip(axes.ravel(), feats_cmp):
    sns.kdeplot(df1[col].dropna(), ax = ax, color = '#2b7a78', label = 'Planta 1', fill = True, alpha = 0.3)
    sns.kdeplot(df2[col].dropna(), ax = ax, color = '#5b3a9e', label = 'Planta 2', fill = True, alpha = 0.3)
    ax.set_title(col)
    ax.legend(fontsize = 8)
fig.suptitle('Sobreposição das distribuições - Planta 1 e Planta 2', fontsize = 14, y = 1.02)
plt.tight_layout()
plt.show()


4.4 Relações comparadas: 

In [ ]:
fig, axes = plt.subplots(1, 2, figsize = (15, 6.5))
sns.heatmap(corr_1, annot = True, fmt = '.2f', cmap = 'coolwarm', vmin = -1, vmax = 1, square = True, ax = axes[0], cbar = False)
axes[0].set_title('Planta 1')
sns.heatmap(corr_2, annot = True, fmt = '.2f', cmap = 'coolwarm', vmin = -1, vmax = 1, square = True, ax = axes[1])
axes[1].set_title('Planta 2')
fig.suptitle('Matrizes de correlação', fontsize = 14, y = 1.02)
plt.tight_layout()
plt.show()


In [ ]:
n_top = min(len(top5_corr_1_tab), len(top5_corr_2_tab))
top5_comp = pd.DataFrame({
    'Ranking': range(1, n_top + 1),
    'Planta 1 - variável': top5_corr_1_tab['variável'].values[:n_top],
    'Planta 1 - correlação': top5_corr_1_tab['correlação_com_AC_POWER'].round(3).values[:n_top],
    'Planta 2 - variável': top5_corr_2_tab['variável'].values[:n_top],
    'Planta 2 - correlação': top5_corr_2_tab['correlação_com_AC_POWER'].round(3).values[:n_top],
})
top5_comp


In [ ]:
fig, axes = plt.subplots(1, 2, figsize = (14, 5.5), sharey = True)
sample1 = df1.sample(min(4000, len(df1)), random_state = RANDOM_STATE)
sample2 = df2.sample(min(4000, len(df2)), random_state = RANDOM_STATE)

sns.scatterplot(data=sample1, x = 'IRRADIATION', y = TARGET, alpha = 0.3, s = 12, ax = axes[0], color = '#2b7a78')
axes[0].set_title(f"Planta 1: IRRADIATION vs AC_POWER (r={corr_1.loc['IRRADIATION', TARGET]:.3f})")

sns.scatterplot(data = sample2, x = 'IRRADIATION', y = TARGET, alpha = 0.3, s = 12, ax = axes[1], color = '#5b3a9e')
axes[1].set_title(f"Planta 2: IRRADIATION vs AC_POWER (r={corr_2.loc['IRRADIATION', TARGET]:.3f})")

fig.suptitle('Relação Irradiação x Potência AC - Planta 1 e Planta 2', fontsize = 13, y = 1.03)
plt.tight_layout()
plt.show()


4.5 Resumo da comparação: 

Ambas as plantas cobrem um período semelhante (aproximadamente de 34 dias, de maio a junho de 2020). 

Os valores ausentes foram tratados com interpolação temporal (Planta 1). 

A variável alvo `AC_POWER` é fortemente concentrad em zero em ambas as plantas, refletindo os períodos noturnos sem geração solar. 

Em ambas as plantas, `DC_POWER` e `IRRADIATION` são as variáveis mais correlacionadas com `AC_POWER`.

As distribuições de temperatura ambiente e do módulo são semelhantes entre as plantas, sugerindo condições climáticas regionais próximas, enquanto a magnitude de `DC_POWER`/`AC_POWER` pode diferir conforme a capacidade instalada de cada planta.